# Data augmentation MODEL

In [ ]:
import os
import numpy as np
import pydicom
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Reshape, Flatten, LeakyReLU
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint

# Define constants
DATA_DIR = r'C:\Users\ASUS\OneDrive\Bureau\AD'
IMAGE_SHAPE = (160, 160)
LATENT_DIM = 100
BATCH_SIZE = 1024
EPOCHS = 100
CHECKPOINT_DIR = r'C:\Users\ASUS\OneDrive\Bureau\model_checkpoints'

# Ensure the checkpoint directory exists
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ModelCheckpoint with save_freq
checkpoint_callback = ModelCheckpoint(
    filepath=os.path.join(CHECKPOINT_DIR, 'gan_epoch_{epoch:04d}.ckpt'),
    save_weights_only=True,
    save_freq='epoch',  # Save every epoch, can be set to an integer for batch frequency
    verbose=1
)


with tf.device('/GPU:0'):
    # Function to load data in batches
    def load_data_batch(data_dir, batch_size, target_shape=(160, 160)):
        for patient_dir in os.listdir(data_dir):
            patient_data = []
            for filename in os.listdir(os.path.join(data_dir, patient_dir)):
                if filename.endswith(".dcm"):
                    dcm_path = os.path.join(data_dir, patient_dir, filename)
                    dcm = pydicom.dcmread(dcm_path)
                    img = dcm.pixel_array
                    img = np.clip((img - np.min(img)) / (np.max(img) - np.min(img) + 1e-5), 0, 1)
                    img = np.resize(img, target_shape)
                    patient_data.append(img)
                    if len(patient_data) == batch_size:
                        yield np.array(patient_data)
                        patient_data = []
            if patient_data:
                yield np.array(patient_data)

    # Generator model
    def build_generator(latent_dim):
        model = Sequential([
            Dense(128, input_dim=latent_dim),
            LeakyReLU(alpha=0.2),
            Dense(np.prod(IMAGE_SHAPE), activation='sigmoid'),
            Reshape(IMAGE_SHAPE)
        ])
        return model

    # Discriminator model
    def build_discriminator(input_shape):
        model = Sequential([
            Flatten(input_shape=input_shape),
            Dense(128),
            LeakyReLU(alpha=0.2),
            Dense(1, activation='sigmoid')
        ])
        return model

        # Build and compile the discriminator
    discriminator = build_discriminator(IMAGE_SHAPE)
    discriminator.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.0002, beta_1=0.5), metrics=['accuracy'])

    # Build the generator
    generator = build_generator(LATENT_DIM)

    # Combined GAN model
    gan = Sequential([generator, discriminator])
    discriminator.trainable = False  # Freeze the discriminator during GAN training
    gan.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.0002, beta_1=0.5))

    
    # Training loop
    for epoch in range(EPOCHS):
        print(f"Epoch: {epoch+1}/{EPOCHS}")
        batch_count = 0  
        for batch_data in load_data_batch(DATA_DIR, BATCH_SIZE, IMAGE_SHAPE):
            batch_count += 1
            # Generate random noise vectors for the generator
            noise = np.random.normal(0, 1, (batch_data.shape[0], LATENT_DIM))
            generated_images = generator.predict(noise)

            
            X = np.concatenate([batch_data, generated_images])
            y_dis = np.zeros(2 * batch_data.shape[0])
            y_dis[:batch_data.shape[0]] = 0.9  

            # Train the discriminator
            discriminator.trainable = True
            d_loss = discriminator.train_on_batch(X, y_dis)

            # Train the generator (via the combined model)
            noise = np.random.normal(0, 1, (BATCH_SIZE, LATENT_DIM))
            y_gen = np.ones(BATCH_SIZE)
            discriminator.trainable = False
            g_loss = gan.train_on_batch(noise, y_gen)

            if batch_count % 10 == 0:  
                print(f"Epoch: {epoch+1}, Batch: {batch_count}, Discriminator Loss: {d_loss}, Generator Loss: {g_loss}")

        
        print(f"Epoch: {epoch+1} completed, Discriminator Loss: {d_loss}, Generator Loss: {g_loss}")
        if (epoch + 1) % 5 == 0:
            generator.save_weights(os.path.join(CHECKPOINT_DIR, f'generator_epoch_{epoch+1}.h5'))
            discriminator.save_weights(os.path.join(CHECKPOINT_DIR, f'discriminator_epoch_{epoch+1}.h5'))
            print(f"Checkpoint saved at epoch {epoch+1}")

    # Function to generate and save DICOM images
    def generate_dicom_images(generator, latent_dim, num_images=5, output_dir='generated_dicom'):
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
        for i in range(num_images):
            noise = np.random.normal(0, 1, (1, latent_dim))
            generated_image = generator.predict(noise)[0]
            pixel_min = np.min(generated_image)
            pixel_max = np.max(generated_image)
            generated_image = (generated_image - pixel_min) * 255 / (pixel_max - pixel_min)
            ds = pydicom.dataset.Dataset()
            ds.PatientName = "Generated Patient"
            ds.PatientID = str(i)
            ds.StudyInstanceUID = generate_uid()
            ds.SeriesInstanceUID = generate_uid()
            ds.SOPInstanceUID = generate_uid()
            ds.Rows = IMAGE_SHAPE[0]
            ds.Columns = IMAGE_SHAPE[1]
            ds.PhotometricInterpretation = 'MONOCHROME2'
            ds.PixelRepresentation = 0
            ds.BitsStored = 8
            ds.BitsAllocated = 8
            ds.HighBit = 7
            ds.SamplesPerPixel = 1
            ds.PixelData = generated_image.astype(np.uint8).tobytes()
            ds.is_little_endian = True
            ds.is_implicit_VR = True
            output_file = os.path.join(output_dir, f"generated_image_{i}.dcm")
            ds.save_as(output_file)
            print(f"Generated DICOM image {i+1}/{num_images}: {output_file}")

    # Generate and save DICOM images
    generate_dicom_images(generator, LATENT_DIM, num_images=1000, output_dir=r'C:\Users\ASUS\OneDrive\Bureau\dcm results')

# Data augmentation MODEL with checkpoint

In [ ]:
import os
import numpy as np
import pydicom
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Reshape, Flatten, LeakyReLU
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint
import datetime

# Define constants
DATA_DIR = r'C:\Users\ASUS\OneDrive\Bureau\AD'
IMAGE_SHAPE = (160, 160)
LATENT_DIM = 100
BATCH_SIZE = 1024
EPOCHS = 100
CHECKPOINT_DIR = r'C:\Users\ASUS\OneDrive\Bureau\model_checkpoints'

# Ensure the checkpoint directory exists
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ModelCheckpoint with save_freq
checkpoint_callback = ModelCheckpoint(
    filepath=os.path.join(CHECKPOINT_DIR, 'gan_epoch_{epoch:04d}.ckpt'),
    save_weights_only=True,
    save_freq='epoch',  # Save every epoch, can be set to an integer for batch frequency
    verbose=1
)

with tf.device('/GPU:0'):
    # Function to load data in batches
    def load_data_batch(data_dir, batch_size, target_shape=(160, 160)):
        for patient_dir in os.listdir(data_dir):
            patient_data = []
            for filename in os.listdir(os.path.join(data_dir, patient_dir)):
                if filename.endswith(".dcm"):
                    dcm_path = os.path.join(data_dir, patient_dir, filename)
                    dcm = pydicom.dcmread(dcm_path)
                    img = dcm.pixel_array
                    img = np.clip((img - np.min(img)) / (np.max(img) - np.min(img) + 1e-5), 0, 1)
                    img = np.resize(img, target_shape)
                    patient_data.append(img)
                    if len(patient_data) == batch_size:
                        yield np.array(patient_data)
                        patient_data = []
            if patient_data:
                yield np.array(patient_data)
    
    # Define the generator model
    def build_generator(latent_dim):
        model = Sequential([
            Dense(128, input_dim=latent_dim),
            LeakyReLU(alpha=0.2),
            Dense(np.prod(IMAGE_SHAPE), activation='sigmoid'),
            Reshape(IMAGE_SHAPE)
        ])
        return model
    
    # Define the discriminator model
    def build_discriminator(input_shape):
        model = Sequential([
            Flatten(input_shape=input_shape),
            Dense(128),
            LeakyReLU(alpha=0.2),
            Dense(1, activation='sigmoid')
        ])
        return model
    
    # Initialize and compile models
    generator = build_generator(LATENT_DIM)
    discriminator = build_discriminator(IMAGE_SHAPE)
    
    # Load weights from the last checkpoint
    generator.load_weights(r'C:\Users\ASUS\OneDrive\Bureau\model_checkpoints\generator_epoch_65.h5')
    discriminator.load_weights(r'C:\Users\ASUS\OneDrive\Bureau\model_checkpoints\discriminator_epoch_65.h5')
    
    discriminator.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.0002, beta_1=0.5), metrics=['accuracy'])
    gan = Sequential([generator, discriminator])
    discriminator.trainable = False
    gan.compile(loss='binary_crossentropy', optimizer=Adam(learning_rate=0.0002, beta_1=0.5))

    # Training loop
    for epoch in range(65, EPOCHS):  # Start from epoch 21
        start_time = datetime.datetime.now()
        print(f"Epoch: {epoch}/{EPOCHS} started at {start_time.strftime('%Y-%m-%d %H:%M:%S')}")
        batch_count = 0
        for batch_data in load_data_batch(DATA_DIR, BATCH_SIZE, IMAGE_SHAPE):
            batch_count += 1
            noise = np.random.normal(0, 1, (batch_data.shape[0], LATENT_DIM))
            generated_images = generator.predict(noise)

            X = np.concatenate([batch_data, generated_images])
            y_dis = np.zeros(2 * batch_data.shape[0])
            y_dis[:batch_data.shape[0]] = 0.9  # One-sided label smoothing
            discriminator.trainable = True
            d_loss = discriminator.train_on_batch(X, y_dis)

            noise = np.random.normal(0, 1, (BATCH_SIZE, LATENT_DIM))
            y_gen = np.ones(BATCH_SIZE)
            discriminator.trainable = False
            g_loss = gan.train_on_batch(noise, y_gen)

            if batch_count % 10 == 0:
                print(f"Batch {batch_count}: Discriminator Loss = {d_loss}, Generator Loss = {g_loss}")

        end_time = datetime.datetime.now()
        print(f"Epoch: {epoch} ended at {end_time.strftime('%Y-%m-%d %H:%M:%S')}")

        if epoch % 5 == 0:
            print(f"Summary for Epoch {epoch}: Started at {start_time.strftime('%Y-%m-%d %H:%M:%S')}, Ended at {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
            generator.save_weights(os.path.join(CHECKPOINT_DIR, f'generator_epoch_{epoch}.h5'))
            discriminator.save_weights(os.path.join(CHECKPOINT_DIR, f'discriminator_epoch_{epoch}.h5'))
            print(f"Checkpoint saved at epoch {epoch}")

from pydicom.uid import generate_uid

    
    # Function to generate and save DICOM images
    def generate_dicom_images(generator, latent_dim, num_images=5, output_dir='generated_dicom'):
        if not os.path.exists(output_dir):
            os.makedirs(output_dir)
        for i in range(num_images):
            noise = np.random.normal(0, 1, (1, latent_dim))
            generated_image = generator.predict(noise)[0]
            pixel_min = np.min(generated_image)
            pixel_max = np.max(generated_image)
            generated_image = (generated_image - pixel_min) * 255 / (pixel_max - pixel_min)
    
            # Prepare DICOM dataset
            ds = pydicom.dataset.Dataset()
            ds.PatientName = "Generated Patient"
            ds.PatientID = str(i)
            ds.StudyInstanceUID = generate_uid()
            ds.SeriesInstanceUID = generate_uid()
            ds.SOPInstanceUID = generate_uid()
            ds.Rows = IMAGE_SHAPE[0]
            ds.Columns = IMAGE_SHAPE[1]
            ds.PhotometricInterpretation = 'MONOCHROME2'
            ds.PixelRepresentation = 0
            ds.BitsStored = 8
            ds.BitsAllocated = 8
            ds.HighBit = 7
            ds.SamplesPerPixel = 1
            ds.PixelData = generated_image.astype(np.uint8).tobytes()
            ds.is_little_endian = True
            ds.is_implicit_VR = True
    
            # Save DICOM file
            output_file = os.path.join(output_dir, f"generated_image_{i}.dcm")
            ds.save_as(output_file)
            print(f"Generated DICOM image {i+1}/{num_images}: {output_file}")
    
    generate_dicom_images(generator, LATENT_DIM, num_images=1000, output_dir=r'C:\Users\ASUS\OneDrive\Bureau\dcm results')


# AD DICOM image generation

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Reshape, Flatten, LeakyReLU
from tensorflow.keras.optimizers import Adam
import pydicom
from pydicom.dataset import Dataset, FileMetaDataset, FileDataset
from pydicom.uid import ExplicitVRLittleEndian
from pydicom.uid import generate_uid
from datetime import datetime

# Define constants
IMAGE_SHAPE = (160, 160)
LATENT_DIM = 100

def build_generator(latent_dim):
    model = Sequential([
        Dense(128, input_dim=latent_dim),
        LeakyReLU(alpha=0.2),
        Dense(np.prod(IMAGE_SHAPE), activation='sigmoid'),
        Reshape(IMAGE_SHAPE)
    ])
    return model

def load_model_weights(model, weight_path):
    """Load weights into the model from a given path."""
    model.load_weights(weight_path)
    print("Weights loaded successfully from:", weight_path)

# Initialize the generator
generator = build_generator(LATENT_DIM)

# Path to the weights file
weights_path = r'C:\Users\Administrator\Downloads\model_checkpoints_jdid\generator_epoch_200.h5'

# Load weights
load_model_weights(generator, weights_path)

def generate_dicom_images(generator, latent_dim, num_patients=5, num_images_per_patient=5, output_dir='generated_dicom'):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    for patient_id in range(num_patients):
        patient_folder = f'patient_{patient_id}'
        patient_dir = os.path.join(output_dir, patient_folder)
        if not os.path.exists(patient_dir):
            os.makedirs(patient_dir)
        
        # Generate a single SeriesInstanceUID for each patient
        series_uid = generate_uid()

        # Generate a base noise vector for the patient
        base_noise = np.random.normal(0, 1, (1, latent_dim))
        
        for i in range(num_images_per_patient):
            # Add small variations to the base noise vector for each image
            noise = base_noise + np.random.normal(0, 0.05, (1, latent_dim))
            generated_image = generator.predict(noise)[0]
            pixel_min = np.min(generated_image)
            pixel_max = np.max(generated_image)
            generated_image = (generated_image - pixel_min) * 255 / (pixel_max - pixel_min)
    
            # Prepare DICOM dataset
            file_meta = pydicom.dataset.FileMetaDataset()
            file_meta.MediaStorageSOPClassUID = pydicom.uid.generate_uid()
            file_meta.MediaStorageSOPInstanceUID = pydicom.uid.generate_uid()
            file_meta.ImplementationClassUID = pydicom.uid.PYDICOM_IMPLEMENTATION_UID
            file_meta.TransferSyntaxUID = ExplicitVRLittleEndian

            ds = FileDataset(f"{patient_folder}_generated_image_{i}.dcm", {}, file_meta=file_meta, preamble=b"\0" * 128)
            ds.PatientName = f"Generated Patient {patient_id}"
            ds.PatientID = str(patient_id)
            ds.StudyInstanceUID = generate_uid()
            ds.SeriesInstanceUID = series_uid  # Same series UID for all images of this patient
            ds.SOPInstanceUID = generate_uid()  # Unique SOP Instance UID
            ds.Rows = 160
            ds.Columns = 160
            ds.PixelSpacing = [1.5, 1.5]
            ds.ImageOrientationPatient = [1, 0, 0, 0, 1, 0]
            ds.ImagePositionPatient = [0.0, 0.0, 67.5 + i * 1.5]  # Increment position for each slice
            ds.SliceThickness = 1.5
            ds.PhotometricInterpretation = 'MONOCHROME2'
            ds.PixelRepresentation = 0
            ds.BitsStored = 8
            ds.BitsAllocated = 8
            ds.HighBit = 7
            ds.SamplesPerPixel = 1
            ds.PixelData = generated_image.astype(np.uint8).tobytes()
            ds.is_little_endian = True
            ds.is_implicit_VR = True
            ds.ContentDate = str(datetime.now().date()).replace('-', '')
            ds.ContentTime = str(datetime.now().time()).replace(':', '').split('.')[0]
    
            # Save DICOM file
            output_file = os.path.join(patient_dir, f"generated_image_{i}.dcm")
            ds.save_as(output_file)
            print(f"Generated DICOM image {i+1}/{num_images_per_patient} for patient {patient_id}: {output_file}")


# Generate DICOM images using the GPU
with tf.device('/GPU:0'):
    generate_dicom_images(generator, LATENT_DIM, num_patients=1000, num_images_per_patient=96, output_dir=r'G:\Generated ad images')